# 🌌 GigaGraph Unified Training Hub (v10.0)

Set `CONFIG['algorithm']` to switch between algorithms:
- `'aptp'` — APTP-GNN v8.12 (3.2B)
- `'noprop'` — NoProp Denoising v9.0 (100M prototype)

In [ ]:
# ── Cell 1: Config ──────────────────────────────────────────────────────────
CONFIG = {
    'algorithm': 'noprop',         # 'aptp' | 'noprop'
    'vocab_size': 128256,
    'lr': 1e-3,
    'warmup_steps': 400,
    'ignore_checkpoint': True,
    'wandb_project': 'gigagraph-v10',
    'wandb_run_id': 'noprop-proto-v10-1',
    # APTP specific
    'aptp_d_model': 3072,
    'aptp_depth': 32,
    'aptp_batch_size': 2,
    'aptp_seq_len': 1024,
    # NoProp specific
    'noprop_d_model': 768,
    'noprop_depth': 8,
    'noprop_batch_size': 4,
    'noprop_seq_len': 512,
}

In [ ]:
# ── Cell 2: Environment Setup ────────────────────────────────────────────────
import os, sys, importlib, torch, wandb, time
from kaggle_secrets import UserSecretsClient

# Silence Git & Sync Source
REPO_URL = 'https://github.com/ey3lock3r/gnn-llm.git'
if not os.path.exists('.git'):
    os.system('git init . > /dev/null 2>&1')
os.system(f'git remote add origin {REPO_URL} > /dev/null 2>&1 || git remote set-url origin {REPO_URL} > /dev/null 2>&1')
os.system('git fetch origin > /dev/null 2>&1 && git reset --hard origin/master > /dev/null 2>&1')

# Authorize W&B and HF from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    os.environ['WANDB_API_KEY'] = user_secrets.get_secret("WANDB_API_KEY")
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
    wandb.login(key=os.environ['WANDB_API_KEY'])
except Exception as e:
    print(f"⚠️ Secrets/W&B Login skipped: {e}")

# Install deps
os.system('pip install -q datasets transformers python-dotenv wandb tqdm > /dev/null 2>&1')

import gnn_llm
importlib.reload(gnn_llm)
from gnn_llm import build_model
from gnn_llm.data.pipeline import GigaDataPipeline
from gnn_llm.training.utils import init_wandb
from gnn_llm.training.trainer import run_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'
algo = CONFIG['algorithm']
print('Algorithm:', algo, '| Device:', device)

In [ ]:
# ── Cell 3: Build Model ──────────────────────────────────────────────────────
if algo == 'aptp':
    model = build_model(
        'aptp',
        vocab_size=CONFIG['vocab_size'],
        depth=CONFIG['aptp_depth'],
        d_model=CONFIG['aptp_d_model'],
        device=device
    )
    batch_size, seq_len = CONFIG['aptp_batch_size'], CONFIG['aptp_seq_len']
elif algo == 'noprop':
    model = build_model(
        'noprop',
        vocab_size=CONFIG['vocab_size'],
        d_model=CONFIG['noprop_d_model'],
        depth=CONFIG['noprop_depth'],
        device=device
    )
    batch_size, seq_len = CONFIG['noprop_batch_size'], CONFIG['noprop_seq_len']
else:
    raise ValueError('Unknown algorithm: ' + algo)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {algo} | Params: {n_params/1e6:.1f}M')

In [ ]:
# ── Cell 4: Run Training ─────────────────────────────────────────────────────
timestamp = time.strftime("%m%d-%H%M%S")
unique_run_id = f"{CONFIG['wandb_run_id']}-{timestamp}"
init_wandb(CONFIG['wandb_project'], unique_run_id)

pipeline = GigaDataPipeline()
loader = pipeline.get_dataloader(batch_size=batch_size, seq_len=seq_len)

print(f'🚀 GigaGraph v10.0 Unified Trainer Launching [Run: {unique_run_id}]...')
run_training(model, loader, CONFIG)
wandb.finish()